# **作业 3 -卷积神经网络**

这是李鸿毅教授机器学习课程作业3的示例代码。

在本作业中，您需要构建一个用于图像分类的卷积神经网络，可能需要一些高级训练技巧。


这里分为三个层次：

**简单**：构建一个简单的卷积神经网络作为基线。 （2分）
**中**：设计更好的架构或采用不同的数据增强来提高性能。 （2分）

**困难**：利用提供的未标记数据来获得更好的结果。 （2分）

## **关于数据集**

这里使用的数据集是 food-11，是 11 个类别的食物图像的集合。

针对作业中的要求，助教稍微修改了数据。
请不要访问原始的完全标记的训练数据或测试标签。

另外，修改后的数据集仅供本课程使用，禁止任何进一步分发或商业用途。

In [1]:
# Download the dataset
# You may choose where to download the data.

# Google Drive
# !gdown --id '1awF7pZ9Dz7X1jn1_QAiKN-_v56veCEKy' --output food-11.zip

# Dropbox
# !wget https://www.dropbox.com/s/m9q6273jl3djall/food-11.zip -O food-11.zip

# MEGA
# !sudo apt install megatools
# !megadl "https://mega.nz/#!zt1TTIhK!ZuMbg5ZjGWzWX1I6nEUbfjMZgCmAgeqJlwDkqdIryfg"

# Unzip the dataset.
# This may take some time.
# !unzip -q food-11.zip

## **Import Packages**

First, we need to import packages that will be used later.

In this homework, we highly rely on **torchvision**, a library of PyTorch.

In [ ]:
# Import necessary packages.
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
# "ConcatDataset" and "Subset" are possibly useful when doing semi-supervised learning.
from torch.utils.data import ConcatDataset, DataLoader, Dataset, Subset
from torchvision.datasets import DatasetFolder

# This is for the progress bar.
from tqdm.auto import tqdm
import torchvision
import os


/Users/Learning/AI/DeepAndReinforced/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## **数据集、数据加载器和转换**

Torchvision 提供了许多有用的实用程序，用于图像预处理、数据包装以及数据增强。

在这里，由于我们的数据通过类标签存储在文件夹中，因此我们可以直接应用 **torchvision.datasets.DatasetFolder**来包装数据，而不需要太多努力。

有关不同变换的详细信息，请参阅[PyTorch官网](https://pytorch.org/vision/stable/transforms.html)。

In [ ]:
# 训练时数据增强 + 归一化
train_tfm = transforms.Compose([
    # 随机缩放裁剪到 128x128，模拟不同距离和构图（比 Resize+Crop 更灵活）
    transforms.RandomResizedCrop((128, 128)),
    # AutoAugment：自动选择最优数据增强策略（ImageNet 策略适合自然图片/食物）
    # transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),
    # 水平翻转，食物左右对称，增加样本多样性
    transforms.RandomHorizontalFlip(),
    # 随机调整亮度/对比度/饱和度，模拟不同光线环境
    # transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    # 转成 PyTorch 张量，值范围 [0, 1]
    transforms.ToTensor(),
    # 标准化：减去均值除以标准差，使输入分布稳定，加速收敛
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    # 随机擦除一部分区域（遮挡），强迫模型关注局部特征，防过拟合
    # transforms.RandomErasing(p=0.3),
])

# 验证/测试只需要 Resize + ToTensor + Normalize（必须和训练保持一致）
test_tfm = transforms.Compose([
    # 固定尺寸到 128x128
    transforms.Resize((128, 128)),
    # 转成 PyTorch 张量
    transforms.ToTensor(),
    # 标准化：必须使用和训练集完全相同的均值和标准差
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [4]:
batch_size = 128

train_set = DatasetFolder("food-11/training/labeled", loader=Image.open, extensions="jpg", transform=train_tfm)
valid_set = DatasetFolder("food-11/validation", loader=Image.open, extensions="jpg", transform=test_tfm)
unlabeled_set = DatasetFolder("food-11/training/unlabeled", loader=Image.open, extensions="jpg", transform=train_tfm)
test_set = DatasetFolder("food-11/testing", loader=Image.open, extensions="jpg", transform=test_tfm)

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

## **型号**

这里的基本模型只是一堆卷积层，后面跟着一些全连接层。

由于彩色图像（RGB）有三个通道，因此网络的输入通道必须是三个。
在每个卷积层中，通常输入的通道会增长，而高度和宽度会缩小（或根据某些超参数（如步幅和填充）保持不变）。
在输入全连接层之前，必须将特征图展平为单个一维向量（对于每个图像）。
然后，这些特征通过全连接层进行转换，最后，我们获得每个类的“logits”。

### **警告——你必须知道**
您可以在此处自由修改模型架构以进一步改进。
但是，如果您想使用一些众所周知的架构（例如 ResNet50），请确保**不要**加载预训练的权重。
使用此类预先训练的模型被视为作弊，因此您将受到惩罚。
同样，如果您使用 **torch.hub**加载任何模块，您有责任确保不使用预先训练的权重。

例如，如果您使用 ResNet-18 作为模型：

model = torchvision.models.resnet18(pretrained=**False**) → 这很好。

model = torchvision.models.resnet18(pretrained=**True**) → 这是**不允许的**。

In [ ]:
class SEBlock(nn.Module):
    """Squeeze-and-Excitation 通道注意力"""
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        w = self.fc(x).unsqueeze(-1).unsqueeze(-1)
        return x * w


class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        # input: [3, 128, 128]
        self.cnn_layers = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            SEBlock(64),
            nn.MaxPool2d(2, 2, 0),  # [64, 64, 64]

            nn.Conv2d(64, 128, 3, 1, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            SEBlock(128),
            nn.MaxPool2d(2, 2, 0),  # [128, 32, 32]

            nn.Conv2d(128, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            SEBlock(256),
            nn.MaxPool2d(2, 2, 0),  # [256, 16, 16]

            nn.Conv2d(256, 512, 3, 1, 1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            SEBlock(512),
            nn.MaxPool2d(2, 2, 0),  # [512, 8, 8]

            nn.Conv2d(512, 512, 3, 1, 1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),  # [512, 4, 4]
        )
        # 512 * 4 * 4 = 8192
        self.fc_layers = nn.Sequential(
            nn.Linear(8192, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 11),
        )

    def forward(self, x):
        x = self.cnn_layers(x)
        x = x.flatten(1)
        x = self.fc_layers(x)
        return x

## **培训**

只需运行提供的代码，无需任何修改，即可完成监督学习。

函数“get_pseudo_labels”用于半监督学习。
如果使用未标记的数据进行半监督学习，预计会获得更好的性能。
但是，您必须自己实现该功能，并且需要手动调整多个超参数。
有关半监督学习的更多详细信息，请参阅[Prof.李的幻灯片](https://speech.ee.ntu.edu.tw/~tlkagk/courses/ML_2016/Lecture/semi%20(v3).pdf)。

再次请注意，**禁止**使用外部数据（或预训练模型）进行训练。

In [ ]:
def get_pseudo_labels(dataset, model, threshold=0.7):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    model.eval()
    # 将 logits 转为概率分布（所有值在 0~1，总和为 1）
    softmax = nn.Softmax(dim=-1)

    selected_indices = []
    current_idx = 0  # 当前 batch 在整个数据集中的起始索引

    for batch in tqdm(data_loader):
        img, _ = batch

        with torch.no_grad():
            logits = model(img.to(device))

        # probs: [batch_size, 11]，每个样本在 11 个类别上的概率
        probs = softmax(logits)
        # max_probs: 每个样本的最大概率（置信度）
        # pseudo_labels: 对应的类别索引（伪标签）
        max_probs, pseudo_labels = probs.max(dim=-1)

        # 生成布尔张量，标记哪些样本的置信度 >= 阈值
        mask = max_probs >= threshold
        for i in range(img.size(0)):
            if mask[i].item():  # .item() 将单元素张量转为 Python bool
                global_idx = current_idx + i  # 计算该样本在 dataset 中的全局索引
                selected_indices.append(global_idx)
                # dataset.samples 格式为 [(路径, 标签), ...]，将假标签替换为伪标签
                path, _ = dataset.samples[global_idx]
                dataset.samples[global_idx] = (path, pseudo_labels[i].item())
        current_idx += img.size(0)  # 更新起始位置，供下一个 batch 使用

    model.train()
    # Subset 返回原始 dataset 中选中的子集，标签已被替换为伪标签
    return Subset(dataset, selected_indices)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = Classifier().to(device)
model.device = device

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=80)

n_epochs = 500
do_semi = True

best_acc = 0.0
patience = 25
no_improve = 0

history = {"train_loss": [], "train_acc": [], "valid_loss": [], "valid_acc": []}

for epoch in range(n_epochs):
    if do_semi and best_acc > 0.7 and epoch % 5 == 0:
        pseudo_set = get_pseudo_labels(unlabeled_set, model)
        concat_dataset = ConcatDataset([train_set, pseudo_set])
        train_loader = DataLoader(concat_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)

    # ---------- 训练 ----------
    model.train()
    train_loss = []
    train_accs = []

    for batch in tqdm(train_loader):
        imgs, labels = batch
        logits = model(imgs.to(device))
        loss = criterion(logits, labels.to(device))

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=10)
        optimizer.step()

        acc = (logits.argmax(dim=-1) == labels.to(device)).float().mean()
        train_loss.append(loss.item())
        train_accs.append(acc.item())

    train_loss = sum(train_loss) / len(train_loss)
    train_acc = sum(train_accs) / len(train_accs)
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    print(f"[ Train | {epoch + 1:03d}/{n_epochs:03d} ] loss = {train_loss:.5f}, acc = {train_acc:.5f}")

    # ---------- 验证 ----------
    model.eval()
    valid_loss = []
    valid_accs = []

    for batch in tqdm(valid_loader):
        imgs, labels = batch
        with torch.no_grad():
            logits = model(imgs.to(device))
        loss = criterion(logits, labels.to(device))
        acc = (logits.argmax(dim=-1) == labels.to(device)).float().mean()
        valid_loss.append(loss.item())
        valid_accs.append(acc.item())

    valid_loss = sum(valid_loss) / len(valid_loss)
    valid_acc = sum(valid_accs) / len(valid_accs)
    history["valid_loss"].append(valid_loss)
    history["valid_acc"].append(valid_acc)
    print(f"[ Valid | {epoch + 1:03d}/{n_epochs:03d} ] loss = {valid_loss:.5f}, acc = {valid_acc:.5f}")

    scheduler.step()

    if valid_acc > best_acc:
        best_acc = valid_acc
        torch.save(model.state_dict(), "best_model.pth")
        no_improve = 0
        print(f"  -> 保存最佳模型 (acc = {best_acc:.5f})")
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"  -> 验证集连续 {patience} 个 epoch 未提升，提前停止。")
            break

model.load_state_dict(torch.load("best_model.pth"))
print(f"\n最佳验证集 acc: {best_acc:.5f}")

# ---------- 绘图 ----------
import matplotlib.pyplot as plt

epochs_range = range(1, len(history["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(epochs_range, history["train_loss"], label="Train Loss")
ax1.plot(epochs_range, history["valid_loss"], label="Valid Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True)

ax2.plot(epochs_range, history["train_acc"], label="Train Acc")
ax2.plot(epochs_range, history["valid_acc"], label="Valid Acc")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig("training_curve.png", dpi=150)
plt.show()

## **测试**

为了进行推理，我们需要确保模型处于 eval 模式，并且数据集的顺序不应打乱（test_loader 中的“shuffle=False”）。

最后但并非最不重要的一点是，不要忘记将预测保存到单个 CSV 文件中。
CSV 文件的格式应遵循幻灯片中提到的规则。

### **警告——记住**

作弊行为包括但不限于：
1.使用测试标签，
2.向往届 Kaggle 比赛提交结果，
3.与他人分享预测，
4.复制地球上任何生物的代码，
5.要求其他人为你做这件事。

任何违规行为都会给你带来惩罚，从最终成绩打折到课程不及格。

您有责任检查您的代码是否违反规则。
当引用网上的代码时，你应该知道这些代码到底是做什么的。
如果您违反规则并声称您不知道这些代码的作用，您将**不会**被容忍。